In [6]:
import sys
import pathlib
from pathlib import Path

# Add project root to Python path
project_root = Path().resolve().parent  # points to repo root
sys.path.insert(0, str(project_root))

from app.environment.tennis_env import TennisEnv
from app.agents.dqn_agent import DQNAgent
from app.training.trainer import Trainer
from app.data.transition_graph import TransitionBuilder


"""Main training script for tennis RL agent"""

# Set up paths
data_path = project_root / "data" / "processed" / "shot_transitions_combined.csv"

print("Building transition graph...")
# Build transition graph
graph_builder = TransitionBuilder(
    transitions_path=str(data_path),
    temperature=1.0
)
transition_graph = graph_builder.build()
print("Transition graph built successfully!")

# Create environment
print("Creating tennis environment...")
env = TennisEnv(
    transition_graph=transition_graph,
    serve_first=True,
    
)

# Create DQN agent
print("Initializing DQN agent...")
agent = DQNAgent(
    env=env,
    lr=0.001,
    gamma=0.95,
    epsilon=1.0,
    epsilon_min=0.01,
    epsilon_decay=0.999,
    memory_size=10000,
    batch_size=32,
    target_update_freq=100
)

# Create trainer
trainer = Trainer(
    env=env,
    agent=agent,
    mlflow_tracking_uri="https://mlflow.digi.com.br",
    experiment_name="tennis-rl-dqn"
)

# Training configuration
training_config = {
    "episodes": 100,
    "save_freq": 100,
    "eval_freq": 200,
    "run_name": "dqn_tennis_v1",
    "tags": {
        "model_type": "DQN",
        "environment": "tennis",
        "data_source": "charting-m-points-2020s",
        "temperature": "1.0"
    }
}

print(f"Starting training for {training_config['episodes']} episodes...")

Building transition graph...
Transition graph built successfully!
Creating tennis environment...
Initializing DQN agent...
Starting training for 100 episodes...


In [26]:
print(agent.state_size)
print(agent.action_size)

29
54


In [30]:
for i in range(300):
    action = agent.act(env.state)
    next_state, reward, done, info = env.step(action)

Ação ilegal detectada: shot_type='p' shot_direction=3
Ação ilegal detectada: shot_type='o' shot_direction=2
Ação ilegal detectada: shot_type='f' shot_direction=3
Ação ilegal detectada: shot_type='k' shot_direction=2
Ação ilegal detectada: shot_type='p' shot_direction=1
Ação ilegal detectada: shot_type='b' shot_direction=3
Ação ilegal detectada: shot_type='b' shot_direction=2
Ação ilegal detectada: shot_type='p' shot_direction=1
Ação ilegal detectada: shot_type='r' shot_direction=2
Ação ilegal detectada: shot_type='k' shot_direction=1
Ação ilegal detectada: shot_type='y' shot_direction=2
Ação ilegal detectada: shot_type='v' shot_direction=1
Ação ilegal detectada: shot_type='r' shot_direction=1
Ação ilegal detectada: shot_type='i' shot_direction=1
Ação ilegal detectada: shot_type='b' shot_direction=2
Ação ilegal detectada: shot_type='v' shot_direction=2
Ação ilegal detectada: shot_type='l' shot_direction=3
Ação ilegal detectada: shot_type='v' shot_direction=1
Vez do Turn.PLAYER
Chosen ne

In [31]:
print(env.state)

last_shot_type='winner' last_shot_direction=3 player_game_score='40' player_set_score=3 pc_game_score='15' pc_set_score=0 player_serves=True


In [32]:
env.state.encode(env)  # Test state encoding

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 3,
 1,
 3,
 0,
 1]

In [29]:
import torchinfo

torchinfo.summary(agent.q_network, input_size=(1, agent.state_size))

Layer (type:depth-idx)                   Output Shape              Param #
DQNNetwork                               [1, 54]                   --
├─Linear: 1-1                            [1, 128]                  3,840
├─Linear: 1-2                            [1, 128]                  16,512
├─Linear: 1-3                            [1, 54]                   6,966
Total params: 27,318
Trainable params: 27,318
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.03
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.11
Estimated Total Size (MB): 0.11